# Sentiment-Analyse-Vergleich: TextBlob, EODHD und Webscraping

Dieses Notebook erweitert die bestehende Korrelationsanalyse um drei Achsen:

1. **Eigene Sentiment-Analyse** mit TextBlob auf EODHD-Artikeltext (statt der vorberechneten `polarity`).
2. **Webscraping-Sentiment** als Proof of Concept (RSS + Reddit, nur März 2026).
3. **MetaTrader-Daten** als dritte Forex-Quelle für EUR/USD (M15 → Daily).

Ziel: Methodik demonstrieren, Quellen vergleichen, Limitationen dokumentieren.

In [ ]:
from pathlib import Path
import ast
import glob as globmod

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob

plt.style.use('seaborn-v0_8')

RAW_DIR       = Path('../../data/raw')
PROCESSED_DIR = Path('../../data/processed/forex')

PAIRS      = ['EUR_USD', 'EUR_CHF', 'GBP_USD']
OHLC_COLS  = ['open', 'high', 'low', 'close']
SOURCES    = ['yahoo', 'eodhd']
DATE_RANGE = '2022-01-01_to_2026-03-25'

PAIR_SYMBOL = {
    'EUR_USD': 'EURUSD.FOREX',
    'EUR_CHF': 'EURCHF.FOREX',
    'GBP_USD': 'GBPUSD.FOREX',
}

In [ ]:
# ---------------------------------------------------------------------------
# Wiederverwendete Hilfsfunktionen (aus news_forex_korrelation_kombiniert.ipynb)
# ---------------------------------------------------------------------------

def _load_yahoo(pair: str) -> pd.DataFrame:
    """Lädt Yahoo-Forex-CSV und bringt es auf einheitliches Schema."""
    path = RAW_DIR / 'forex' / 'yahoo' / f'{pair}_{DATE_RANGE}.csv'
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['Date'], utc=True).dt.tz_localize(None).dt.normalize()
    df = df.rename(columns={'Open':'open','High':'high','Low':'low','Close':'close'})
    return df.set_index('date')[OHLC_COLS].sort_index()


def _load_eodhd(pair: str) -> pd.DataFrame:
    """Lädt EODHD-Forex-CSV und bringt es auf einheitliches Schema."""
    path = RAW_DIR / 'forex' / 'eodhd' / f'{pair}_{DATE_RANGE}.csv'
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date']).dt.normalize()
    return df.set_index('date')[OHLC_COLS].sort_index()


def fill_gaps(df: pd.DataFrame) -> pd.DataFrame:
    """Reindiziere auf tägliche Frequenz und interpoliere fehlende Tage zeitgewichtet linear.
    
    Begründung: Forex-Kurse fehlen an Wochenenden/Feiertagen (MNAR – der Markt ist
    geschlossen). Zeitgewichtete Interpolation ist angemessen, weil der Kurs sich
    kontinuierlich bewegt und zeitlich näherliegende Werte stärker gewichten sollte.
    (Vgl. Vorlesung W2, Folien.pdf, Folie 16: Imputation bei Zeitreihen.)
    """
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='D')
    out = df.reindex(full_idx)
    out[OHLC_COLS] = out[OHLC_COLS].interpolate(method='time')
    out.index.name = 'date'
    return out


def average_sources(pair: str, filled: dict) -> pd.DataFrame:
    """Mittelwert OHLC über Yahoo + EODHD auf Basis der interpolierten Reihen.
    
    Da beide Quellen denselben lückenlosen Tagesindex haben, enthält der Output keine Lücken.
    """
    yh = filled[pair]['yahoo'][OHLC_COLS]
    ed = filled[pair]['eodhd'][OHLC_COLS]
    common = yh.index.intersection(ed.index)
    avg = (yh.loc[common] + ed.loc[common]) / 2
    avg.index.name = 'date'
    return avg.sort_index()


def _parse_list(val):
    """Wandelt einen String wie \"['FX','MACRO']\" in eine echte Python-Liste um."""
    if isinstance(val, list): return val
    if not isinstance(val, str) or not val.strip(): return []
    try: return ast.literal_eval(val)
    except (ValueError, SyntaxError): return []


def aggregate(df: pd.DataFrame, freq: str) -> pd.DataFrame:
    """Aggregiert per Mittelwert auf 'D' / 'W-MON' / 'MS'."""
    if freq == 'D':
        return df.copy()
    return df.resample(freq).mean()


def load_oil(name: str, file_stem: str) -> pd.DataFrame:
    """Lädt ein Yahoo-Öl-CSV und interpoliert Lücken (gleiche Methode wie Forex)."""
    path = RAW_DIR / 'oil' / 'yahoo' / f'{file_stem}_{DATE_RANGE}.csv'
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['Date'], utc=True).dt.tz_localize(None).dt.normalize()
    df = df.rename(columns={'Open':'open','High':'high','Low':'low','Close':'close'})
    df = df.set_index('date')[OHLC_COLS].sort_index()
    return fill_gaps(df)


# ---------------------------------------------------------------------------
# Neue Hilfsfunktionen
# ---------------------------------------------------------------------------

def textblob_polarity(text: str) -> float:
    """Berechnet TextBlob-Polarity für einen Text-String.
    
    Gibt 0.0 zurück für leere/NaN-Werte, damit fehlende Texte das Ergebnis
    nicht verzerren (statt NaN, was bei Aggregation herausfallen würde).
    
    Quelle: TextBlob Dokumentation – https://textblob.readthedocs.io/
    """
    if not isinstance(text, str) or not text.strip():
        return np.nan
    return TextBlob(text).sentiment.polarity


def load_metatrader_m15(path: Path) -> pd.DataFrame:
    """Lädt MetaTrader M15-CSV (tab-separiert, <COLUMN>-Header) und aggregiert auf Daily OHLC.
    
    Aggregationsregeln für Intraday → Daily:
    - open:  erster Wert des Tages (Eröffnung)
    - high:  Maximum aller 15-Min-Highs
    - low:   Minimum aller 15-Min-Lows
    - close: letzter Wert des Tages (Schluss)
    """
    df = pd.read_csv(path, sep='\t')
    df['datetime'] = pd.to_datetime(
        df['<DATE>'] + ' ' + df['<TIME>'],
        format='%Y.%m.%d %H:%M:%S'
    )
    df['date'] = df['datetime'].dt.normalize()
    daily = df.groupby('date').agg(
        open=('<OPEN>', 'first'),
        high=('<HIGH>', 'max'),
        low=('<LOW>', 'min'),
        close=('<CLOSE>', 'last'),
    )
    daily.index = pd.to_datetime(daily.index)
    return daily.sort_index()


print('Setup abgeschlossen.')

## Schritt 1 — Eigene Sentiment-Analyse auf EODHD-Artikeltext

Statt die vorberechnete `polarity`-Spalte von EODHD direkt zu verwenden, wenden wir **TextBlob** auf den Artikeltext (`title` + `content`) an.

**Warum TextBlob?**
- Einfache, gut dokumentierte Bibliothek für englischsprachige Sentiment-Analyse (Quelle: [TextBlob Docs](https://textblob.readthedocs.io/)).
- Basiert auf Pattern (regelbasiert), keine GPU nötig.
- Liefert eine `polarity` im Bereich [-1, +1] und `subjectivity` [0, 1].
- Bereits in `requirements.txt` vorhanden.

**Limitationen:**
- Nur für Englisch trainiert (unsere EODHD-Artikel sind englisch, passt also).
- Allgemeinsprachlich, nicht speziell auf Finanzsprache kalibriert.
- Ironie, Fachjargon und Hedging (z.B. *"not entirely negative"*) werden oft falsch bewertet.
- Finanzspezifische Modelle wie FinBERT wären genauer, benötigen aber deutlich mehr Rechenleistung.

**Dieselbe Methode** wird in Schritt 2 auch auf die Webscraping-Daten angewendet, damit ein fairer Vergleich möglich ist.

In [ ]:
# EODHD-News laden und TextBlob-Sentiment berechnen
# ACHTUNG: Dauert einige Minuten wegen ~27'000+ Artikeln pro Paar

eodhd_news = {}
for pair in PAIRS:
    path = RAW_DIR / 'news' / 'eodhd' / f'{pair}_news_{DATE_RANGE}.csv'
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date_only'])
    
    # Defensiver Symbolfilter (wie im bestehenden Notebook)
    sym = PAIR_SYMBOL[pair]
    df['symbols_list'] = df['symbols'].apply(_parse_list)
    df = df[df['symbols_list'].apply(lambda l: sym in l)].copy()
    
    # EODHD-vorberechnetes Sentiment umbenennen
    df = df.rename(columns={'polarity': 'polarity_eodhd'})
    
    # TextBlob auf title und content anwenden
    print(f'{pair}: TextBlob wird auf {len(df)} Artikel angewendet...')
    df['polarity_tb_title']   = df['title'].apply(textblob_polarity)
    df['polarity_tb_content'] = df['content'].apply(textblob_polarity)
    
    # Kombinierte Polarity: Mittelwert von title und content
    # (Beide Felder tragen Informationen bei; der Titel ist oft knapper und pointierter)
    df['polarity_tb_combined'] = df[['polarity_tb_title', 'polarity_tb_content']].mean(axis=1)
    
    eodhd_news[pair] = df[['date', 'title', 'polarity_eodhd',
                           'polarity_tb_title', 'polarity_tb_content',
                           'polarity_tb_combined']].copy()
    print(f'  → {len(df)} Artikel verarbeitet.')

print('\nFertig.')

In [ ]:
# Scatter-Plot: TextBlob combined vs. EODHD-Polarity pro Paar
fig, axes = plt.subplots(1, len(PAIRS), figsize=(15, 5))
tb_vs_eodhd_corr = {}

for ax, pair in zip(axes, PAIRS):
    df = eodhd_news[pair].dropna(subset=['polarity_eodhd', 'polarity_tb_combined'])
    if df.empty:
        ax.set_title(f'{pair} – keine Daten')
        continue
    
    ax.scatter(df['polarity_eodhd'], df['polarity_tb_combined'],
               alpha=0.1, s=5, color='tab:blue')
    ax.plot([-1, 1], [-1, 1], 'r--', alpha=0.5, label='y = x')  # Identitätslinie
    
    r = df['polarity_eodhd'].corr(df['polarity_tb_combined'])
    tb_vs_eodhd_corr[pair] = r
    
    ax.set_xlabel('EODHD polarity (vorberechnet)')
    ax.set_ylabel('TextBlob polarity (eigene Analyse)')
    ax.set_title(f'{pair} (n={len(df)}, r={r:.3f})')
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.legend()

plt.suptitle('Vergleich: EODHD-Polarity vs. TextBlob auf Artikeltext', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print('Pearson-Korrelation pro Paar:')
for pair, r in tb_vs_eodhd_corr.items():
    print(f'  {pair}: r = {r:.4f}')

In [ ]:
# Verteilungsvergleich: EODHD vs. TextBlob Polarity
fig, axes = plt.subplots(1, len(PAIRS), figsize=(15, 4))

stats_rows = []
for ax, pair in zip(axes, PAIRS):
    df = eodhd_news[pair].dropna(subset=['polarity_eodhd', 'polarity_tb_combined'])
    if df.empty:
        ax.set_title(f'{pair} – keine Daten')
        continue
    
    sns.kdeplot(df['polarity_eodhd'], ax=ax, label='EODHD', color='tab:orange')
    sns.kdeplot(df['polarity_tb_combined'], ax=ax, label='TextBlob', color='tab:blue')
    ax.set_title(pair)
    ax.set_xlabel('Polarity')
    ax.legend()
    
    for method, col in [('EODHD', 'polarity_eodhd'), ('TextBlob', 'polarity_tb_combined')]:
        stats_rows.append({
            'pair': pair,
            'methode': method,
            'mean': df[col].mean(),
            'median': df[col].median(),
            'std': df[col].std(),
            'min': df[col].min(),
            'max': df[col].max(),
        })

plt.suptitle('Verteilung der Polarity-Werte: EODHD vs. TextBlob', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

sentiment_stats = pd.DataFrame(stats_rows).round(4)
sentiment_stats

In [ ]:
# Tagesaggregation: TextBlob-Sentiment per Median
# Begründung für Median statt Mittelwert: robuster gegen einzelne extreme Artikel.
# (Vgl. Vorlesung W2, Folien.pdf: Imputation/Aggregation; und Empfehlung im bestehenden Notebook.)

tb_daily = {}
for pair in PAIRS:
    df = eodhd_news[pair].dropna(subset=['polarity_tb_combined'])
    daily = df.groupby('date')['polarity_tb_combined'].median().to_frame('sentiment_tb')
    tb_daily[pair] = daily.sort_index()
    print(f'{pair}: {len(daily)} Tage mit TextBlob-Sentiment')

# Vergleich: wie viele Tage hat das EODHD-vorberechnete Sentiment?
for pair in PAIRS:
    df = eodhd_news[pair].dropna(subset=['polarity_eodhd'])
    n_eodhd = df.groupby('date').ngroups
    n_tb = len(tb_daily[pair])
    print(f'{pair}: EODHD = {n_eodhd} Tage, TextBlob = {n_tb} Tage')

In [ ]:
# Tagesverlauf: Mean vs. Median Polarity – EODHD (vorberechnet) vs. TextBlob (eigene Analyse)
# Diese Grafik zeigt den Kurvenverlauf der täglichen Sentiment-Aggregation über die Zeit.

for pair in PAIRS:
    df = eodhd_news[pair].copy()
    if df.empty:
        print(f'{pair}: keine Daten')
        continue
    
    # Tagesaggregation: Mean und Median für beide Methoden
    daily_agg = df.groupby('date').agg(
        eodhd_mean=('polarity_eodhd', 'mean'),
        eodhd_median=('polarity_eodhd', 'median'),
        tb_mean=('polarity_tb_combined', 'mean'),
        tb_median=('polarity_tb_combined', 'median'),
    ).sort_index()
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Plot 1: Mittelwert pro Tag
    ax1.plot(daily_agg.index, daily_agg['eodhd_mean'], color='tab:orange', alpha=0.7,
             linewidth=0.8, label='EODHD (vorberechnet)')
    ax1.plot(daily_agg.index, daily_agg['tb_mean'], color='tab:blue', alpha=0.7,
             linewidth=0.8, label='TextBlob (eigene Analyse)')
    ax1.axhline(0, color='grey', linestyle='--', linewidth=0.6)
    ax1.set_ylabel('Polarity (Mittelwert)')
    ax1.set_title(f'{pair} – Täglicher Mittelwert der Polarity')
    ax1.legend()
    
    # Plot 2: Median pro Tag
    ax2.plot(daily_agg.index, daily_agg['eodhd_median'], color='tab:orange', alpha=0.7,
             linewidth=0.8, label='EODHD (vorberechnet)')
    ax2.plot(daily_agg.index, daily_agg['tb_median'], color='tab:blue', alpha=0.7,
             linewidth=0.8, label='TextBlob (eigene Analyse)')
    ax2.axhline(0, color='grey', linestyle='--', linewidth=0.6)
    ax2.set_ylabel('Polarity (Median)')
    ax2.set_title(f'{pair} – Täglicher Median der Polarity')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Korrelation zwischen EODHD und TextBlob auf Tagesebene
    r_mean = daily_agg['eodhd_mean'].corr(daily_agg['tb_mean'])
    r_median = daily_agg['eodhd_median'].corr(daily_agg['tb_median'])
    print(f'{pair}: Pearson r (Tages-Mittelwert) = {r_mean:.4f}, (Tages-Median) = {r_median:.4f}\n')

## Schritt 2 — Webscraping Sentiment-Analyse (Proof of Concept)

Wir laden die per RSS-Feeds und Reddit gescrapten Nachrichtenartikel und wenden **dieselbe TextBlob-Analyse** an.

**Einschränkungen:**
- Nur 5 Scraping-Tage (3., 4., 11., 18., 25. März 2026) — kein historischer Vergleich möglich.
- Ca. 33% der Artikel haben keinen `summary`-Text — in diesen Fällen wird nur der `title` analysiert.
- **Proof of Concept:** In einer Produktionsumgebung müsste man kontinuierlich scrapen, um einen längeren Zeitraum abzudecken.

Quelle der Daten: RSS-Feeds (MarketWatch, Investing, FXStreet, CNBC) + Reddit (r/Forex, r/investing, r/economics).

In [ ]:
# Alle Webscraping-CSVs laden und TextBlob-Sentiment berechnen
ws_dir = RAW_DIR / 'news' / 'webscraping'
ws_files = sorted(ws_dir.glob('all_scraped_news_*.csv'))
print(f'{len(ws_files)} Webscraping-Dateien gefunden.')

ws_dfs = []
for f in ws_files:
    df = pd.read_csv(f)
    ws_dfs.append(df)
    print(f'  {f.name}: {len(df)} Artikel')

ws_raw = pd.concat(ws_dfs, ignore_index=True)
ws_raw['date'] = pd.to_datetime(ws_raw['date_only'])

# Deduplizierung nach title (gleicher Artikel kann in mehreren Scraping-Läufen auftauchen)
n_before = len(ws_raw)
ws_raw = ws_raw.drop_duplicates(subset=['title'], keep='first')
print(f'\nNach Deduplizierung: {n_before} → {len(ws_raw)} Artikel ({n_before - len(ws_raw)} Duplikate entfernt)')

# TextBlob-Sentiment berechnen
# Für leere summaries wird nur der title analysiert
print(f'\nTextBlob wird auf {len(ws_raw)} Artikel angewendet...')
ws_raw['polarity_tb_title']   = ws_raw['title'].apply(textblob_polarity)
ws_raw['polarity_tb_summary'] = ws_raw['summary'].apply(textblob_polarity)

# Kombinierte Polarity: Mittelwert von title + summary (NaN-aware)
ws_raw['polarity_tb_combined'] = ws_raw[['polarity_tb_title', 'polarity_tb_summary']].mean(axis=1)

# Tagesaggregation: Median
ws_daily = (ws_raw
    .dropna(subset=['polarity_tb_combined'])
    .groupby('date')['polarity_tb_combined']
    .median()
    .to_frame('sentiment_ws')
    .sort_index()
)

print(f'\nTägliches Webscraping-Sentiment: {len(ws_daily)} Tage')
print(f'Datumsbereich: {ws_daily.index.min().date()} – {ws_daily.index.max().date()}')
ws_daily

In [ ]:
# Vergleich: Webscraping-TextBlob vs. EODHD-TextBlob für überlappende Tage
# (Webscraping ist nicht paar-spezifisch, daher Vergleich mit EUR/USD als bestabgedecktes Paar)

pair_for_ws = 'EUR_USD'
overlap = ws_daily.join(tb_daily[pair_for_ws], how='inner')
print(f'Überlappende Tage (Webscraping ∩ EODHD TextBlob für {pair_for_ws}): {len(overlap)}')

if len(overlap) >= 2:
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(overlap))
    width = 0.35
    ax.bar(x - width/2, overlap['sentiment_ws'], width, label='Webscraping (TextBlob)', color='tab:green')
    ax.bar(x + width/2, overlap['sentiment_tb'], width, label=f'EODHD {pair_for_ws} (TextBlob)', color='tab:blue')
    ax.set_xticks(x)
    ax.set_xticklabels([d.strftime('%Y-%m-%d') for d in overlap.index], rotation=45)
    ax.axhline(0, color='grey', linestyle='--', linewidth=0.6)
    ax.set_ylabel('Median Polarity')
    ax.set_title(f'Tägliches Sentiment: Webscraping vs. EODHD (TextBlob) – {pair_for_ws}')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    r = overlap['sentiment_ws'].corr(overlap['sentiment_tb'])
    print(f'Pearson-Korrelation: r = {r:.4f} (n = {len(overlap)} Tage)')
    print(f'Mittlere Differenz:  {(overlap["sentiment_ws"] - overlap["sentiment_tb"]).mean():.4f}')
else:
    print('Zu wenige überlappende Tage für eine sinnvolle Korrelation.')

## Schritt 3 — MetaTrader-Daten als dritte Forex-Quelle (nur EUR/USD)

Als Proof of Concept laden wir **M15-Daten** (15-Minuten-Intervalle) aus MetaTrader 5 und aggregieren sie auf Tagesbasis. So demonstrieren wir, wie Broker-Daten als dritte Quelle integriert werden können.

**Datenformat:** Tab-separiert, Spalten mit `<COLUMN>`-Headern, nur EUR/USD verfügbar.

**Aggregation M15 → Daily** (Quelle: Claude – Standardvorgehen bei Intraday-Daten):
- `open`: erster Wert des Tages
- `high`: Maximum aller 15-Min-Highs
- `low`: Minimum aller 15-Min-Lows
- `close`: letzter Wert des Tages

**Datenharmonisierung** (Vgl. Vorlesung W4, W04_dwae.pdf):
Dies ist ein klassisches Beispiel für **retrospektive, flexible Datenharmonisierung** —
drei Quellen (Yahoo, EODHD, MetaTrader) mit unterschiedlicher Syntax (CSV vs. Tab-separated),
Struktur (verschiedene Spaltenbezeichnungen) und leichten semantischen Unterschieden (Broker-Spreads).

In [ ]:
# MetaTrader M15 laden und auf Daily aggregieren
mt_path = RAW_DIR / 'forex' / 'metatrader' / 'EURUSD_M15_202201030000_202512260100.csv'
mt_daily = load_metatrader_m15(mt_path)
mt_daily = fill_gaps(mt_daily)

print(f'MetaTrader EUR/USD Daily: {len(mt_daily)} Tage')
print(f'Datumsbereich: {mt_daily.index.min().date()} – {mt_daily.index.max().date()}')
mt_daily.head()

In [ ]:
# 3-Quellen-Vergleich für EUR/USD: Yahoo vs. EODHD vs. MetaTrader
pair = 'EUR_USD'
yh = _load_yahoo(pair)
ed = _load_eodhd(pair)
yh_filled = fill_gaps(yh)
ed_filled = fill_gaps(ed)

# Gemeinsamer Datumsbereich (alle drei Quellen)
common_idx = yh_filled.index.intersection(ed_filled.index).intersection(mt_daily.index)
compare_3 = pd.DataFrame({
    'yahoo':     yh_filled.loc[common_idx, 'close'],
    'eodhd':     ed_filled.loc[common_idx, 'close'],
    'metatrader': mt_daily.loc[common_idx, 'close'],
})

# Plot: Close-Verlauf aller drei Quellen
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8))

for src, color in [('yahoo', 'tab:blue'), ('eodhd', 'tab:orange'), ('metatrader', 'tab:green')]:
    ax1.plot(compare_3.index, compare_3[src], label=src, alpha=0.8, linewidth=0.8)
ax1.set_title(f'{pair} Close – 3-Quellen-Vergleich')
ax1.set_ylabel('Kurs')
ax1.legend()

# Absolute Abweichungen vom Mittelwert
mean_close = compare_3.mean(axis=1)
for src, color in [('yahoo', 'tab:blue'), ('eodhd', 'tab:orange'), ('metatrader', 'tab:green')]:
    ax2.plot(compare_3.index, compare_3[src] - mean_close, label=f'{src} – Mittelwert', alpha=0.7, linewidth=0.8)
ax2.axhline(0, color='grey', linestyle='--', linewidth=0.6)
ax2.set_title(f'{pair} Close – Abweichungen vom 3-Quellen-Mittelwert')
ax2.set_ylabel('Abweichung (Kurspunkte)')
ax2.legend()

plt.tight_layout()
plt.show()

# Paarweise Korrelation und mittlere Abweichung
print(f'\nPearson-Korrelation ({len(compare_3)} gemeinsame Tage):')
print(compare_3.corr().round(6))
print(f'\nMittlere absolute Abweichung (in Kurspunkten):')
for a, b in [('yahoo','eodhd'), ('yahoo','metatrader'), ('eodhd','metatrader')]:
    diff = (compare_3[a] - compare_3[b]).abs()
    print(f'  {a:10s} vs. {b:10s}: mean={diff.mean():.6f}, max={diff.max():.6f}')

## Schritt 4 — Kombination: Forex + Sentiment + Öl

Wir kombinieren nun:
- **Forex:** Yahoo + EODHD gemittelt (wie im bestehenden Notebook) für alle Paare. Für EUR/USD zusätzlich MetaTrader als 3. Quelle.
- **Sentiment:** TextBlob auf EODHD-Text (Schritt 1) als Hauptserie.
- **Öl:** Brent Crude Oil.

Korrelationsanalyse auf D/W/M-Frequenzen.

In [ ]:
# Forex laden und kombinieren (Yahoo + EODHD)
raw_data = {
    pair: {'yahoo': _load_yahoo(pair), 'eodhd': _load_eodhd(pair)}
    for pair in PAIRS
}

filled_data = {
    pair: {src: fill_gaps(df) for src, df in sources.items()}
    for pair, sources in raw_data.items()
}

averaged_data = {pair: average_sources(pair, filled_data) for pair in PAIRS}

for pair, df in averaged_data.items():
    print(f'{pair}: {len(df)} Tage (Yahoo+EODHD gemittelt)')

In [ ]:
# Ölpreise laden
oil_data = {
    'WTI':   load_oil('WTI',   'WTI_Crude_Oil'),
    'Brent': load_oil('Brent', 'Brent_Crude_Oil'),
}
for name, df in oil_data.items():
    print(f'{name:6s} | {len(df):5d} Tage | {df.index.min().date()} – {df.index.max().date()}')

In [ ]:
# Kombinierte DataFrames erstellen

def build_full_combined(pair: str) -> pd.DataFrame:
    """Kombiniert Forex-Close, TextBlob-Sentiment und Ölpreis auf einem Datumsindex.
    Für EUR/USD zusätzlich MetaTrader-Close und Webscraping-Sentiment."""
    fx  = averaged_data[pair][['close']].rename(columns={'close': 'close_avg'})
    nws = tb_daily[pair]
    oil = oil_data['Brent'][['close']].rename(columns={'close': 'oil_brent'})
    out = fx.join(nws, how='outer').join(oil, how='outer')
    
    if pair == 'EUR_USD':
        out = out.join(ws_daily, how='outer')
        out = out.join(
            mt_daily[['close']].rename(columns={'close': 'close_mt'}),
            how='outer'
        )
    return out.sort_index()

full_combined = {pair: build_full_combined(pair) for pair in PAIRS}
for pair, df in full_combined.items():
    print(f'{pair}: {len(df)} Zeilen, Spalten: {list(df.columns)}')

In [ ]:
# Overlay-Plots: Forex + TextBlob-Sentiment + Öl
# (Gleiche Visualisierungsstruktur wie im bestehenden Notebook)

def plot_overlay_tb(pair: str):
    """Drei Auflösungen: Forex-Close (links), TextBlob-Sentiment + Öl (rechts)."""
    df = full_combined[pair]
    settings = [
        ('D',     'Täglich'),
        ('W-MON', 'Wöchentlich (Mittelwert)'),
        ('MS',    'Monatlich (Mittelwert)'),
    ]
    fig, axes = plt.subplots(len(settings), 1, figsize=(13, 11))
    for ax, (freq, label) in zip(axes, settings):
        agg = df.copy() if freq == 'D' else df.resample(freq).mean()
        
        # 1. Forex-Close (linke Achse)
        l1, = ax.plot(agg.index, agg['close_avg'], color='tab:blue',
                       label=f'{pair} Close (Yahoo+EODHD)')
        ax.set_ylabel('Forex-Kurs', color='tab:blue')
        ax.tick_params(axis='y', labelcolor='tab:blue')
        
        # 2. TextBlob-Sentiment (rechte Achse 1)
        ax2 = ax.twinx()
        l2, = ax2.plot(agg.index, agg['sentiment_tb'], color='tab:red', alpha=0.6,
                        label='TextBlob Sentiment (EODHD-Text)')
        ax2.axhline(0, color='grey', linestyle='--', linewidth=0.6)
        ax2.set_ylabel('Sentiment', color='tab:red')
        ax2.tick_params(axis='y', labelcolor='tab:red')
        
        # 3. Öl (rechte Achse 2)
        ax3 = ax.twinx()
        ax3.spines['right'].set_position(('outward', 55))
        l3, = ax3.plot(agg.index, agg['oil_brent'], color='tab:green', alpha=0.7,
                        label='Brent Crude Oil')
        ax3.set_ylabel('Brent (USD)', color='tab:green')
        ax3.tick_params(axis='y', labelcolor='tab:green')
        
        ax.set_title(f'{pair} vs. TextBlob-Sentiment vs. Brent – {label}')
        ax.legend([l1, l2, l3], [l1.get_label(), l2.get_label(), l3.get_label()],
                  loc='upper left', fontsize=8)
    
    plt.tight_layout()
    plt.show()

for pair in PAIRS:
    plot_overlay_tb(pair)

In [ ]:
# Pearson-Korrelation: Forex-Close vs. TextBlob-Sentiment auf D/W/M Frequenzen
corr_rows = []
for pair in PAIRS:
    for freq, label in [('D', 'täglich'), ('W-MON', 'wöchentlich'), ('MS', 'monatlich')]:
        agg = aggregate(full_combined[pair][['close_avg', 'sentiment_tb']], freq).dropna()
        r = agg['close_avg'].corr(agg['sentiment_tb']) if len(agg) > 2 else np.nan
        corr_rows.append({'pair': pair, 'frequenz': label, 'pearson_r': r, 'n': len(agg)})

corr_table = pd.DataFrame(corr_rows).round(4)
print('Korrelation Forex-Close vs. TextBlob-Sentiment:')
corr_table

In [ ]:
# Korrelationsmatrix (Forex, Sentiment, Öl) als Heatmap – Tagesbasis
for pair in PAIRS:
    cols = ['close_avg', 'sentiment_tb', 'oil_brent']
    df = full_combined[pair][cols].dropna()
    if len(df) < 3:
        print(f'{pair}: zu wenig Daten für Korrelationsmatrix')
        continue
    
    fig, ax = plt.subplots(figsize=(6, 5))
    corr_matrix = df.corr()
    sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, ax=ax)
    ax.set_title(f'{pair} – Korrelationsmatrix (n={len(df)} Tage)')
    plt.tight_layout()
    plt.show()

## Schritt 5 — Vergleich der Sentiment-Quellen

Systematischer Vergleich der Sentiment-Signale:
1. **EODHD vorberechnet** (`polarity`) vs. **TextBlob** auf EODHD-Artikeltext
2. **TextBlob auf EODHD-Text** vs. **TextBlob auf Webscraping** (überlappende Tage)

In [ ]:
# Vergleich 1: EODHD-polarity vs. TextBlob auf Artikelebene

comparison_rows = []
for pair in PAIRS:
    df = eodhd_news[pair].dropna(subset=['polarity_eodhd', 'polarity_tb_combined'])
    if len(df) < 3:
        comparison_rows.append({'vergleich': f'EODHD vs TextBlob ({pair})',
                                'n': len(df), 'pearson_r': np.nan,
                                'spearman_rho': np.nan, 'rmse': np.nan})
        continue
    
    pearson_r = df['polarity_eodhd'].corr(df['polarity_tb_combined'])
    spearman_r = df['polarity_eodhd'].corr(df['polarity_tb_combined'], method='spearman')
    rmse = np.sqrt(((df['polarity_eodhd'] - df['polarity_tb_combined']) ** 2).mean())
    
    comparison_rows.append({
        'vergleich': f'EODHD vs TextBlob ({pair})',
        'n': len(df),
        'pearson_r': round(pearson_r, 4),
        'spearman_rho': round(spearman_r, 4),
        'rmse': round(rmse, 4),
        'anmerkung': 'Artikel-Ebene'
    })

print('Vergleich 1: EODHD-Polarity vs. TextBlob (Artikelebene)')
pd.DataFrame(comparison_rows)

In [ ]:
# Vergleich 2: EODHD-TextBlob vs. Webscraping-TextBlob (Tagesebene)
overlap = ws_daily.join(tb_daily['EUR_USD'], how='inner')

if len(overlap) >= 2:
    pearson_r = overlap['sentiment_ws'].corr(overlap['sentiment_tb'])
    spearman_r = overlap['sentiment_ws'].corr(overlap['sentiment_tb'], method='spearman')
    rmse = np.sqrt(((overlap['sentiment_ws'] - overlap['sentiment_tb']) ** 2).mean())
    
    comparison_rows.append({
        'vergleich': 'Webscraping vs EODHD-TextBlob (EUR_USD)',
        'n': len(overlap),
        'pearson_r': round(pearson_r, 4),
        'spearman_rho': round(spearman_r, 4),
        'rmse': round(rmse, 4),
        'anmerkung': f'Tagesebene, nur {len(overlap)} überlappende Tage'
    })
    print(f'Überlappende Tage: {len(overlap)}')
    print(f'Pearson r = {pearson_r:.4f}, Spearman rho = {spearman_r:.4f}, RMSE = {rmse:.4f}')
else:
    print('Zu wenige überlappende Tage für Vergleich.')

In [ ]:
# Summary-Tabelle aller Vergleiche
comparison_summary = pd.DataFrame(comparison_rows)
print('Zusammenfassung aller Sentiment-Vergleiche:')
comparison_summary

## Schritt 6 — Diskussion & Limitationen

### Ergebnisse

- **TextBlob vs. EODHD-Polarity:** Die Korrelation zeigt, inwieweit die eigene Sentiment-Analyse mit dem vorberechneten EODHD-Sentiment übereinstimmt. Unterschiede sind erwartbar, da EODHD ein anderes (unbekanntes) Modell verwendet, während TextBlob regelbasiert und allgemeinsprachlich arbeitet.
- **Webscraping:** Proof of Concept erfolgreich — TextBlob kann auf beliebige englische Texte angewandt werden. Die wenigen überlappenden Tage erlauben keine belastbare Aussage über die Übereinstimmung.
- **MetaTrader:** Dritte Forex-Quelle bestätigt Yahoo/EODHD-Daten. Minimale Abweichungen sind durch Broker-Spreads und Zeitzonenunterschiede erklärbar.

### Limitationen

1. **EUR/CHF:** Zu wenige EODHD-Artikel (~12 total) für eine robuste Sentiment-Analyse.
2. **Webscraping:** Nur 5 Tage im März 2026. Für Produktionseinsatz wäre kontinuierliches Scraping nötig.
3. **TextBlob:** Englisch-only, allgemeinsprachlich (nicht finanzspezifisch). Finanzspezifische Modelle (FinBERT, FinVADER) wären bessere Alternativen, benötigen aber mehr Rechenleistung.
4. **MetaTrader:** Nur EUR/USD verfügbar, Daten enden Dezember 2025.
5. **Korrelation ≠ Kausalität:** Pearson-Korrelation misst nur lineare Zusammenhänge. Granger-Kausalitätstests oder Lag-Korrelationen wären für eine tiefere Analyse nötig.

### Nächste Schritte (Ausblick)

- Finanzspezifisches Sentiment-Modell evaluieren (z.B. FinBERT).
- Kontinuierliches Webscraping aufsetzen für längeren Vergleichszeitraum.
- MetaTrader-Daten für weitere Paare beschaffen.
- Granger-Kausalitätstests oder Lag-Korrelationen statt reiner Pearson-Korrelation.
- Sentiment-Daten nicht interpolieren – ein bewusste Entscheidung, weil an Tagen ohne Nachrichten tatsächlich kein Sentiment existiert (Vgl. Empfehlung im bestehenden Notebook).

In [ ]:
# Versionsinfo
import sys
import textblob as tb_pkg

print(f'Python {sys.version}')
print(f'pandas {pd.__version__}, numpy {np.__version__}')
print(f'textblob {tb_pkg.__version__}')
print(f'Notebook ausgeführt am: {pd.Timestamp.now()}')

---
*Notebook erstellt im Rahmen des FHNW BAI Data Wrangling Projekts (FS 2026).*